# Demo 12 — Una sola copia e un eco con feedback

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
nota, sr = al.load_audio(AUDIO / "Guitar_C4.wav", sr=SR, seconds=1.2)
x_echo = np.pad(nota, (0, round(3*sr)))
una_copia = al.delay_signal(0.7 * x_echo, 0.375, gain=0.3, sr=sr)[:len(x_echo)]
ripetizioni = al.delay_by(al.feedback_delay(x_echo, 0.375, feedback=0.55, sr=sr), 0.375, sr=sr)[:len(x_echo)]
eco = 0.7 * x_echo + 0.3 * ripetizioni
al.compare_audio([("Diretto", x_echo), ("Una copia", una_copia),
                  ("Ripetizioni con feedback", eco)], sr)
fig = go.Figure()
passo = 40
fig.add_trace(go.Scatter(x=np.arange(0, len(eco), passo)/sr, y=eco[::passo],
                        name="Diretto + ripetizioni"))
fig.add_vline(x=len(nota)/sr, line_dash="dot", annotation_text="Fine ingresso")
fig.update_xaxes(title_text="Tempo [s]")
fig.update_yaxes(title_text="Ampiezza")
fig.update_layout(height=320, title="Delay di una croma puntata a 120 BPM")
fig.show()
print(f"Picco negli ultimi 200 ms: {al.to_db(al.peak_rms(eco[-sr//5:])[0]):.1f} dBFS")
